# Main Analysis Pipeline

Core analytical code for:  
**"A Large Language Model as a Psychometric Instrument for Occupational Stress Monitoring: A 15-Day Ecological Momentary Assessment Study"**

## Analyses

| Section | Manuscript Reference |
|---|---|
| 4. Extraction repeatability (ICC) | Table 2 |
| 5. Convergent validity (LLM vs LIWC) | Table 3, Supplementary Table 2 |
| 6. Ecological concurrent validity (GEE) | Table 4 |
| 7. Incremental explanatory value (Ridge CV) | Table 5 |
| 8. Predictive validity (Logistic CV) | Table 6 |
| 9. Moderation analysis | Table 7 |

## 13 Psycholinguistic Constructs

| Variable | Construct |
|---|---|
| `llm_posemo` | Positive emotion |
| `llm_negemo` | Negative emotion |
| `llm_anxiety` | Anxiety |
| `llm_anger` | Anger |
| `llm_sadness` | Sadness |
| `llm_cogproc` | Cognitive processing |
| `llm_tentativeness` | Tentativeness |
| `llm_self_focus` | Self-focus |
| `llm_social` | Social references |
| `llm_work` | Work-related content |
| `llm_time_pressure` | Time pressure |
| `llm_somatic` | Somatic complaints |
| `llm_coping` | Coping |

---
## 1. Configuration

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Dict

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy import stats as sp_stats

from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LogisticRegressionCV
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, brier_score_loss,
)

# --- Column names (Korean identifiers from source data) ---
ID_COL = "participant ID"
DAY_COL = "day"
STRESS_COL = "stress"                          # daily EMA stress (0-10)
WC_COL = "word_count"
PSS_V1 = "baseline PSS-10"
PSS_V2 = "follow-up PSS-10"
PHQ_COL = "baseline PHQ-9"

CAT_COVARS = [
    "Sex", "Marriage Status",
    "religion",
    "cohabitation",
    "alcohol use",
    "smoking",
    "is_holiday",
]

BFI_COLS = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Neuroticism",
    "Openness"
]
BFI_LABELS = dict(zip(BFI_COLS, ["Extraversion", "Agreeableness", "Conscientiousness", "Neuroticism", "Openness"]))

NUM_COVARS = ["Age", WC_COL, PHQ_COL] + BFI_COLS

LLM_FEATURES = [
    "llm_posemo", "llm_negemo", "llm_anxiety", "llm_anger", "llm_sadness",
    "llm_cogproc", "llm_tentativeness", "llm_self_focus", "llm_social",
    "llm_work", "llm_time_pressure", "llm_somatic", "llm_coping",
]

---
## 2. Data Loading & Sample Selection

The original participant-level data are not publicly available (see manuscript Data Availability).  
The dataset was pre-filtered for analytic eligibility (see Methods).

In [ ]:
long_df = pd.read_csv("long_df.csv")  # placeholder path

# Exclude participants with fewer than 3 completed EMA days
counts = long_df.groupby(ID_COL).size()
long_df = long_df[~long_df[ID_COL].isin(counts[counts < 3].index)].reset_index(drop=True)

print(f"Participants: {long_df[ID_COL].nunique()}, Person-days: {len(long_df)}")
# Expected output: Participants: 279, Person-days: 3556

---
## 3. Utility Functions

In [ ]:
def safe_zscore(s: pd.Series) -> pd.Series:
    """Z-score with zero-variance guard."""
    s = pd.to_numeric(s, errors="coerce")
    mu, sd = s.mean(), s.std(ddof=0)
    if sd is None or np.isnan(sd) or sd < 1e-12:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd


def add_within_between(df: pd.DataFrame, features: List[str]) -> pd.DataFrame:
    """Within/between decomposition (Curran & Bauer, 2011)."""
    out = df.copy()
    for f in features:
        pm = out.groupby(ID_COL)[f].transform("mean")
        out[f"{f}_w"] = out[f] - pm
        out[f"{f}_b"] = pm
    return out


def standardize_wb(df: pd.DataFrame, features: List[str]) -> pd.DataFrame:
    """Z-score within and between components separately."""
    out = df.copy()
    for f in features:
        for suf in ["_w", "_b"]:
            col = f"{f}{suf}"
            if col in out.columns:
                out[col] = safe_zscore(out[col])
    return out


def cov_formula(num_covars: List[str], include_day: bool = True) -> str:
    """Build covariate formula string for statsmodels."""
    cat = " + ".join(f"C({c})" for c in CAT_COVARS)
    num = " + ".join(num_covars) if num_covars else ""
    day = DAY_COL if include_day else ""
    return " + ".join(p for p in [cat, num, day] if p.strip())


def prepare_analysis_df(long_df: pd.DataFrame, features: List[str], min_obs: int = 2) -> pd.DataFrame:
    """Prepare day-level analysis dataframe with within/between decomposition."""
    df = long_df.copy()

    df[ID_COL] = df[ID_COL].astype(str)
    df[DAY_COL] = pd.to_numeric(df[DAY_COL], errors="coerce")
    df[STRESS_COL] = pd.to_numeric(df[STRESS_COL], errors="coerce")
    for c in CAT_COVARS:
        df[c] = df[c].astype("category")
    for c in list(set(NUM_COVARS)) + features:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=[STRESS_COL] + features + list(set(NUM_COVARS)) + [DAY_COL])
    cnt = df.groupby(ID_COL).size()
    df = df[df[ID_COL].isin(cnt[cnt >= min_obs].index)].copy()

    df = add_within_between(df, features)
    df = standardize_wb(df, features)
    return df

---
## 4. Extraction Repeatability (ICC)

**Table 2.** ICC(2,k) from two-way random-effects model (Shrout & Fleiss, 1979).  
200 text entries x 5 repeated extractions under identical model settings.

In [ ]:
def compute_icc2k(wide: np.ndarray, k: int, alpha: float = 0.05) -> Dict:
    """
    ICC(2,k) with 95% CI via Spearman-Brown transformation.

    Parameters
    ----------
    wide : (n_subjects, k_raters) array
    k : number of raters
    """
    n = wide.shape[0]
    gm = np.mean(wide)
    rm = np.mean(wide, axis=1)
    cm = np.mean(wide, axis=0)

    MS_R = k * np.sum((rm - gm) ** 2) / (n - 1)
    MS_C = n * np.sum((cm - gm) ** 2) / (k - 1)
    SS_tot = np.sum((wide - gm) ** 2)
    MS_E = (SS_tot - k * np.sum((rm - gm) ** 2) - n * np.sum((cm - gm) ** 2)) / ((n - 1) * (k - 1))

    icc21 = (MS_R - MS_E) / (MS_R + (k - 1) * MS_E + k * (MS_C - MS_E) / n)
    icc2k = k * icc21 / (1 + (k - 1) * icc21)

    F_val = MS_R / MS_E
    df1, df2 = n - 1, (n - 1) * (k - 1)
    F_L = F_val / sp_stats.f.ppf(1 - alpha / 2, df1, df2)
    F_U = F_val * sp_stats.f.ppf(1 - alpha / 2, df2, df1)
    vn = k * (MS_C - MS_E) / (n * MS_R) if MS_R > 0 else 0

    sb = lambda r: k * r / (1 + (k - 1) * r) if r > -1 / (k - 1) else np.nan
    lo21 = (F_L - 1) / (F_L + k - 1 + k * vn)
    hi21 = (F_U - 1) / (F_U + k - 1 + k * vn)

    return {
        "ICC_2k": icc2k, "CI_lo": sb(lo21), "CI_hi": sb(hi21),
        "SEM": np.std(rm, ddof=1) * np.sqrt(max(0.0, 1.0 - icc2k)),
    }

# NOTE: Requires 5 repeated extraction datasets (not publicly available).
# For each feature, pivot to (n_subjects x 5) and call compute_icc2k().

---
## 5. Convergent Validity (LLM vs LIWC)

**Table 3 & Supplementary Table 2.** Day-level and person-level Spearman correlations with bootstrap CIs.

In [ ]:
LIWC_MAP = {
    "posemo": "emo_pos", "negemo": "emo_neg", "anxiety": "emo_anx",
    "anger": "emo_anger", "sadness": "emo_sad", "cogproc": "cogproc",
    "tentativeness": "tentat", "self_focus": "i", "social": "Social",
    "work": "work", "time_pressure": "time",
    "somatic": "health",   # proxy: LIWC health category
    "coping": None,         # composite: mean(affiliation, prosocial, achieve, -conflict)
}


def spearman_boot_ci(x, y, n_boot=1200, seed=42):
    """Spearman rho with nonparametric bootstrap 95% CI."""
    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    n = len(tmp)
    if n < 10:
        return np.nan, np.nan, np.nan, n
    rng = np.random.default_rng(seed)
    rho = tmp["x"].rank().corr(tmp["y"].rank())
    boots = [tmp["x"].iloc[b := rng.choice(n, n, replace=True)].rank().corr(
             tmp["y"].iloc[b].rank()) for _ in range(n_boot)]
    lo, hi = np.nanquantile(boots, [0.025, 0.975])
    return rho, lo, hi, n


def convergent_validity(df, llm_features, liwc_col_map):
    """Day-level and person-level Spearman rho with bootstrap CIs."""
    pairs = [(f"llm_{f}", f"liwc_{f}") for f in liwc_col_map]
    rows = []

    for llm_c, liwc_c in pairs:
        feat = llm_c.replace("llm_", "")
        rho, lo, hi, n = spearman_boot_ci(df[llm_c], df[liwc_c])
        rows.append({"feature": feat, "level": "day", "rho": rho, "ci_lo": lo, "ci_hi": hi, "N": n})

    person = df.groupby(ID_COL)[[c for c, _ in pairs] + [c for _, c in pairs]].mean()
    for llm_c, liwc_c in pairs:
        feat = llm_c.replace("llm_", "")
        rho, lo, hi, n = spearman_boot_ci(person[llm_c], person[liwc_c], n_boot=2500, seed=43)
        rows.append({"feature": feat, "level": "person", "rho": rho, "ci_lo": lo, "ci_hi": hi, "N": n})

    return pd.DataFrame(rows)

---
## 6. Ecological Concurrent Validity (GEE)

**Table 4.** GEE with exchangeable correlation structure.  
Within-person and between-person components entered simultaneously.  
FDR correction (Benjamini-Hochberg) across all 26 terms (13 features x 2 levels).

In [ ]:
def fit_gee_single(df, feature, num_covars):
    """GEE for a single feature with within/between decomposition."""
    cov = cov_formula(num_covars)
    formula = f"{STRESS_COL} ~ {cov} + {feature}_w + {feature}_b"
    res = smf.gee(
        formula, groups=df[ID_COL], data=df,
        cov_struct=sm.cov_struct.Exchangeable(), family=sm.families.Gaussian(),
    ).fit()
    return [{
        "feature": feature, "term": t,
        "beta": res.params.get(t, np.nan),
        "se": res.bse.get(t, np.nan),
        "p": res.pvalues.get(t, np.nan),
        "n_obs": int(res.nobs),
    } for t in [f"{feature}_w", f"{feature}_b"]]


def run_gee_all(df, features, num_covars):
    """GEE for all features with FDR correction."""
    rows = []
    for f in features:
        rows.extend(fit_gee_single(df, f, num_covars))
    res = pd.DataFrame(rows)
    mask = res["p"].notna()
    if mask.sum() > 0:
        _, q, _, _ = multipletests(res.loc[mask, "p"].values, method="fdr_bh")
        res.loc[mask, "q_fdr"] = q
    res["ci_low"] = res["beta"] - 1.96 * res["se"]
    res["ci_high"] = res["beta"] + 1.96 * res["se"]
    return res.sort_values("q_fdr").reset_index(drop=True)


# --- Run ---
df_analysis = prepare_analysis_df(long_df, LLM_FEATURES)
gee_results = run_gee_all(df_analysis, LLM_FEATURES, NUM_COVARS)
print(gee_results[["feature", "term", "beta", "se", "q_fdr", "ci_low", "ci_high"]].to_string(index=False))

---
## 7. Incremental Explanatory Value (Ridge CV)

**Table 5.** 5-fold GroupKFold (by participant) ridge regression.  
Baseline (demographics + PHQ-9 + BFI) vs. Baseline + LLM features.

In [ ]:
def build_X(df, features, num_covars, include_llm=True):
    """Design matrix for ridge regression."""
    X_cat = pd.get_dummies(df[CAT_COVARS], drop_first=True)
    X_num = df[num_covars].copy()
    X_day = df[[DAY_COL]].copy()
    blocks = [X_cat, X_num, X_day]
    if include_llm:
        feat_cols = [f"{f}_w" for f in features] + [f"{f}_b" for f in features]
        blocks.append(df[feat_cols])
    return pd.concat(blocks, axis=1)


def run_ridge_cv(df, features, num_covars, n_splits=5):
    """GroupKFold Ridge: baseline vs baseline+LLM."""
    X_base = build_X(df, features, num_covars, include_llm=False)
    X_full = build_X(df, features, num_covars, include_llm=True)
    y = df[STRESS_COL].astype(float)
    groups = df[ID_COL]
    alphas = np.logspace(-3, 3, 25)
    gkf = GroupKFold(n_splits=n_splits)
    records = []
    for fold, (tr, te) in enumerate(gkf.split(X_full, y, groups), 1):
        for label, X in [("baseline", X_base), ("baseline+LLM", X_full)]:
            sc = StandardScaler()
            X_tr, X_te = sc.fit_transform(X.iloc[tr]), sc.transform(X.iloc[te])
            model = RidgeCV(alphas=alphas, cv=5).fit(X_tr, y.iloc[tr])
            pred = model.predict(X_te)
            records.append({"fold": fold, "model": label,
                            "R2": r2_score(y.iloc[te], pred),
                            "MAE": mean_absolute_error(y.iloc[te], pred),
                            "RMSE": np.sqrt(mean_squared_error(y.iloc[te], pred))})
    cv = pd.DataFrame(records)
    return cv, cv.groupby("model")[["R2", "MAE", "RMSE"]].agg(["mean", "std"])


# --- Run ---
ridge_cv, ridge_summary = run_ridge_cv(df_analysis, LLM_FEATURES, NUM_COVARS)
print(ridge_summary)

---
## 8. Predictive Validity (Logistic Regression CV)

**Table 6.** L2-regularized logistic regression with StratifiedKFold.  
Three arms: baseline, LLM-only, baseline+LLM.  
Thresholds: PSS-10 >= 17 and >= 19.

In [ ]:
def coef_of_variation(x):
    x = x.dropna()
    if len(x) < 2 or x.mean() == 0:
        return np.nan
    return x.std() / x.mean()


def prepare_person_df(long_df, features):
    """Aggregate to person-level: mean + CV for LLM features and word count."""
    df = long_df.copy()
    for c in features + [WC_COL, PSS_V1, PSS_V2, PHQ_COL] + BFI_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    agg = {}
    for f in features:
        agg[f"{f}_mean"] = (f, "mean")
        agg[f"{f}_cv"] = (f, coef_of_variation)
    agg["word_count_mean"] = (WC_COL, "mean")
    agg["word_count_cv"] = (WC_COL, coef_of_variation)
    agg["n_days"] = (features[0], "count")
    agg_df = df.groupby(ID_COL).agg(**agg).reset_index()

    fixed = [ID_COL, PSS_V1, PSS_V2, PHQ_COL, "Age"] + CAT_COVARS[:6] + BFI_COLS
    fixed = [c for c in fixed if c in df.columns]
    person = df.groupby(ID_COL)[fixed].first().reset_index(drop=True)
    person = person.merge(agg_df, on=ID_COL, how="inner")
    person = person[person["n_days"] >= 2].copy()
    for c in person.columns:
        if c.endswith("_cv"):
            person[c] = person[c].replace([np.inf, -np.inf], np.nan)
    return person


def make_logistic_X(pdf, features, arm):
    """Design matrix for logistic regression (3 arms)."""
    cat = [c for c in CAT_COVARS[:6] if c in pdf.columns]
    X_cat = pd.get_dummies(pdf[cat], drop_first=True)
    X_demo = pdf[["Age", PHQ_COL] + BFI_COLS].copy()
    llm_cols = [f"{f}_{g}" for f in features for g in ["mean", "cv"]]
    wc_cols = ["word_count_mean", "word_count_cv"]
    X_llm = pdf[[c for c in llm_cols + wc_cols if c in pdf.columns]].copy()

    if arm == "baseline":
        X = pd.concat([X_cat, X_demo], axis=1)
        sc = list(X_demo.columns)
    elif arm == "LLM_only":
        X = X_llm.copy()
        sc = list(X_llm.columns)
    elif arm == "baseline+LLM":
        X = pd.concat([X_cat, X_demo, X_llm], axis=1)
        sc = list(X_demo.columns) + list(X_llm.columns)
    else:
        raise ValueError(arm)
    return X, sc


def run_logistic_cv(pdf, features, outcome, n_splits=5):
    """StratifiedKFold logistic CV for 3 arms."""
    y = pdf[outcome].astype(int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    folds = list(skf.split(np.zeros(len(y)), y))
    records = []
    for arm in ["baseline", "LLM_only", "baseline+LLM"]:
        X, sc_cols = make_logistic_X(pdf, features, arm)
        for fold, (tr, te) in enumerate(folds, 1):
            Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
            for c in Xtr.columns:
                med = Xtr[c].median()
                Xtr[c] = Xtr[c].fillna(med if not pd.isna(med) else 0)
                Xte[c] = Xte[c].fillna(med if not pd.isna(med) else 0)
            scaler = StandardScaler()
            sc = [c for c in sc_cols if c in Xtr.columns]
            Xtr[sc] = scaler.fit_transform(Xtr[sc])
            Xte[sc] = scaler.transform(Xte[sc])
            lr = LogisticRegressionCV(
                Cs=np.logspace(-4, 1, 12), cv=5, scoring="roc_auc",
                penalty="l2", class_weight="balanced", max_iter=5000, random_state=42,
            ).fit(Xtr.values, y.iloc[tr])
            prob = lr.predict_proba(Xte.values)[:, 1]
            yt = y.iloc[te]
            records.append({"fold": fold, "arm": arm,
                            "AUROC": roc_auc_score(yt, prob) if yt.nunique() >= 2 else np.nan,
                            "AUPRC": average_precision_score(yt, prob) if yt.nunique() >= 2 else np.nan,
                            "Brier": brier_score_loss(yt, prob)})
    return pd.DataFrame(records)


# --- Run ---
person_df = prepare_person_df(long_df, LLM_FEATURES)
for cutoff in [17, 19]:
    person_df[f"PSS_v2_ge{cutoff}"] = (pd.to_numeric(person_df[PSS_V2], errors="coerce") >= cutoff).astype(int)

for cutoff in [17, 19]:
    res = run_logistic_cv(person_df, LLM_FEATURES, f"PSS_v2_ge{cutoff}")
    print(f"\n=== PSS >= {cutoff} ===")
    print(res.groupby("arm")[["AUROC", "AUPRC", "Brier"]].agg(["mean", "std"]))

---
## 9. Moderation Analysis

**Table 7.** GEE with interaction terms.  
Moderators: depressive symptom severity (PHQ-9 >= 10) and Big Five traits.  
FDR correction across interaction terms.

In [ ]:
def fit_gee_moderation(df, feature, mod_term, num_covars):
    """GEE with feature x moderator interaction."""
    cov = cov_formula(num_covars)
    fw, fb = f"{feature}_w", f"{feature}_b"
    formula = (f"{STRESS_COL} ~ {cov} + {fw} + {fb} + {mod_term} "
               f"+ {fw}:{mod_term} + {fb}:{mod_term}")
    return smf.gee(
        formula, groups=df[ID_COL], data=df,
        cov_struct=sm.cov_struct.Exchangeable(), family=sm.families.Gaussian(),
    ).fit()


def run_moderation(df, features, mod_term, mod_key, num_covars):
    """Run moderation for all features, extract interaction terms, apply FDR."""
    rows = []
    for f in features:
        try:
            res = fit_gee_moderation(df, f, mod_term, num_covars)
            for t in res.params.index:
                if ":" in t and mod_key in t:
                    rows.append({"feature": f, "term": t,
                                 "beta": float(res.params[t]), "se": float(res.bse[t]),
                                 "p": float(res.pvalues[t]), "n_obs": int(res.nobs)})
        except Exception:
            pass
    out = pd.DataFrame(rows)
    if not out.empty:
        mask = out["p"].notna()
        if mask.sum() > 0:
            _, q, _, _ = multipletests(out.loc[mask, "p"].values, method="fdr_bh")
            out.loc[mask, "q_fdr"] = q
        out["ci_low"] = out["beta"] - 1.96 * out["se"]
        out["ci_high"] = out["beta"] + 1.96 * out["se"]
    return out.sort_values("q_fdr").reset_index(drop=True)


# --- PHQ-9 >= 10 moderation ---
df_mod = df_analysis.copy()
df_mod["PHQ10plus"] = (pd.to_numeric(df_mod[PHQ_COL], errors="coerce") >= 10).astype(int).astype("category")
covars_phq = ["Age", WC_COL] + BFI_COLS

phq_mod = run_moderation(df_mod, LLM_FEATURES, "C(PHQ10plus)", "PHQ10plus", covars_phq)
print("=== PHQ-9 >= 10 (FDR < 0.05) ===")
print(phq_mod[phq_mod["q_fdr"] < 0.05][["feature", "term", "beta", "se", "q_fdr"]].to_string(index=False))

# --- Big Five moderation ---
for trait in BFI_COLS:
    df_t = df_mod.copy()
    tz = f"{trait}_z"
    df_t[tz] = safe_zscore(df_t[trait])
    covars_bfi = ["Age", WC_COL, PHQ_COL] + [c for c in BFI_COLS if c != trait]
    result = run_moderation(df_t, LLM_FEATURES, tz, tz, covars_bfi)
    sig = result[result["q_fdr"] < 0.05]
    if not sig.empty:
        print(f"\n=== {BFI_LABELS[trait]} (FDR < 0.05) ===")
        print(sig[["feature", "term", "beta", "se", "q_fdr"]].to_string(index=False))

---
## References

- Curran, P.J. & Bauer, D.J. (2011). The disaggregation of within-person and between-person effects. *Annual Review of Psychology*, 62, 583-619.
- Shrout, P.E. & Fleiss, J.L. (1979). Intraclass correlations: Uses in assessing rater reliability. *Psychological Bulletin*, 86(2), 420-428.
- Benjamini, Y. & Hochberg, Y. (1995). Controlling the false discovery rate. *JRSS-B*, 57, 289-300.